# Uncensored LLM API Server (vLLM + Cloudflare Tunnel)

Notebook ini menjalankan model LLM uncensored dengan **vLLM** sebagai **OpenAI-compatible API** (`/v1/chat/completions`) yang bisa langsung dipakai dari aplikasi, script, IDE, atau chat client via URL publik sementara (**Cloudflare Tunnel**) yang dilindungi **API Key**.

### Cara Pakai:
1. Pilih GPU dulu di menu **Runtime → Change runtime type**:
   - **L4 (24 GB)** → paling hemat compute units (~2–3 CU/jam), cocok untuk model **24B–27B AWQ**.
   - **A100 (40 GB)** → cocok untuk **24B–32B AWQ / FP8** (~10–15 CU/jam).
   - **H100 (80 GB)** → cocok untuk kelas berat **70B AWQ / FP8** (~18–25+ CU/jam).
2. Pilih model di dropdown **Step 2**, atur opsi jika perlu.
3. Klik **Runtime → Run all** (atau jalankan cell satu per satu dari atas).
4. Copy **Base URL**, **API Key**, dan **Model Name** dari output **Step 4**.

> **Penting:** Setelah selesai dipakai, selalu klik **Runtime → Disconnect and delete runtime** agar Google Colab Compute Units tidak terus berkurang!

In [ ]:
#@title 1. Cek GPU & VRAM Colab
import subprocess, torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU tidak terdeteksi! Buka Runtime -> Change runtime type -> pilih L4, A100, atau H100.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"GPU terdeteksi : {gpu_name}")
print(f"Total VRAM     : {vram_gb:.1f} GB")

if vram_gb >= 75:
    print("Rekomendasi    : Muat semua preset termasuk kelas berat 70B (AWQ / FP8)!")
elif vram_gb >= 38:
    print("Rekomendasi    : Cocok untuk preset [L4/A100 24GB+] dan [A100/H100 40GB+] (24B-32B).")
elif vram_gb >= 22:
    print("Rekomendasi    : Cocok untuk preset [L4/A100 24GB+] (24B-27B AWQ).")
else:
    print("Peringatan     : VRAM < 22 GB. Gunakan CUSTOM_MODEL_ID ukuran 8B-14B (AWQ/FP8).")

print("-" * 60)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]).decode().strip())

In [ ]:
#@title 2. Pilih Model & Konfigurasi API
import os, secrets, torch

MODEL_PRESET = "[L4/A100 24GB+] Valdemardi/Dolphin3.0-Mistral-24B-AWQ" #@param ["[L4/A100 24GB+] Valdemardi/Dolphin3.0-Mistral-24B-AWQ", "[L4/A100 24GB+] Valdemardi/Dolphin3.0-R1-Mistral-24B-AWQ", "[L4/A100 24GB+] alexxorm/Huihui-Qwen3.6-27B-abliterated-AWQ", "[L4/A100 24GB+] noneUsername/Qwen3-32B-abliterated-awq", "[A100/H100 40GB+] dphn/Dolphin-Mistral-24B-Venice-Edition-FP8", "[A100/H100 40GB+] munekazu/Huihui-Qwen3.8-27B-abliterated-FP8", "[H100 80GB] jiangchengchengNLP/llama3.3-70B-instruct-abliterated-awq", "[H100 80GB] cyankiwi/Hermes-4-70B-AWQ-4bit", "[H100 80GB] m8than/huihui-ai-Llama-3.3-70B-Instruct-abliterated-FP8", "[H100 80GB] NousResearch/Hermes-3-Llama-3.1-70B-FP8"]
CUSTOM_MODEL_ID = "" #@param {type:"string"}
SERVED_MODEL_NAME = "" #@param {type:"string"}
MAX_MODEL_LEN = "8192" #@param ["4096", "8192", "16384", "32768", "auto"]
GPU_MEMORY_UTILIZATION = 0.90 #@param {type:"slider", min:0.75, max:0.96, step:0.01}
ENABLE_TOOL_CALLING = False #@param {type:"boolean"}
API_KEY = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

# Ambil Hugging Face repo ID dari preset atau custom input
MODEL_ID = CUSTOM_MODEL_ID.strip() if CUSTOM_MODEL_ID.strip() else MODEL_PRESET.split("] ", 1)[1].strip()
MODEL_ALIAS = SERVED_MODEL_NAME.strip() if SERVED_MODEL_NAME.strip() else MODEL_ID
VLLM_API_KEY = API_KEY.strip() if API_KEY.strip() else f"sk-colab-{secrets.token_urlsafe(24)}"
VLLM_PORT = 8000

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

# Validasi VRAM sederhana
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
if not CUSTOM_MODEL_ID.strip():
    if "[H100 80GB]" in MODEL_PRESET and vram_gb < 70:
        print(f"PERINGATAN: Model 70B butuh ~40-75 GB VRAM (H100 80GB), tapi GPU saat ini hanya {vram_gb:.1f} GB! Kemungkinan besar akan OOM.")
    elif "[A100/H100 40GB+]" in MODEL_PRESET and vram_gb < 35:
        print(f"PERINGATAN: Model FP8 24B-27B butuh ~28-35 GB VRAM (A100/H100), tapi GPU saat ini hanya {vram_gb:.1f} GB! Pilih preset [L4/A100 24GB+] agar aman.")

print(f"Model Repo  : {MODEL_ID}")
print(f"Model Name  : {MODEL_ALIAS}")
print(f"Max Context : {MAX_MODEL_LEN}")
print(f"GPU Util    : {GPU_MEMORY_UTILIZATION}")
print(f"API Key     : {VLLM_API_KEY}")

In [ ]:
#@title 3. Install vLLM, hf_transfer & Cloudflare Tunnel
import os, shutil, subprocess

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Menginstall vLLM & hf_transfer (biasanya 1-3 menit)...")
subprocess.run(
    ["pip", "install", "-q", "--upgrade", "vllm", "hf_transfer", "requests"],
    check=True,
)

if not shutil.which("cloudflared"):
    print("Mengunduh cloudflared...")
    subprocess.run(
        [
            "curl", "-fsSL",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", "/usr/local/bin/cloudflared",
        ],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

vllm_ver = subprocess.check_output(["python3", "-c", "import vllm; print(vllm.__version__)"]).decode().strip()
cf_ver = subprocess.check_output(["cloudflared", "--version"]).decode().strip()
print(f"Siap! vLLM {vllm_ver} | {cf_ver}")

In [ ]:
#@title 4. Jalankan vLLM API Server + Cloudflare Tunnel
import os, re, time, subprocess, requests

# Hentikan proses lama jika sebelumnya sudah jalan
subprocess.run(["pkill", "-f", "vllm serve"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)
time.sleep(2)

VLLM_LOG = "/tmp/vllm.log"
CF_LOG = "/tmp/cloudflared.log"

cmd = [
    "vllm", "serve", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--api-key", VLLM_API_KEY,
    "--served-model-name", MODEL_ALIAS,
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--dtype", "auto",
    "--trust-remote-code",
]

if ENABLE_TOOL_CALLING:
    lower_id = MODEL_ID.lower()
    parser = "hermes"
    if "mistral" in lower_id:
        parser = "mistral"
    elif "llama-3" in lower_id or "llama3" in lower_id:
        parser = "llama3_json"
    cmd += ["--enable-auto-tool-choice", "--tool-call-parser", parser]

if MAX_MODEL_LEN != "auto":
    cmd += ["--max-model-len", str(MAX_MODEL_LEN)]

env = os.environ.copy()
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Menjalankan vLLM server (download + load model ke GPU)...")
print("Command:", " ".join(cmd).replace(VLLM_API_KEY, "***"))

vllm_log_f = open(VLLM_LOG, "w")
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log_f, stderr=subprocess.STDOUT, env=env)

# Tunggu vLLM siap (maksimal 20 menit untuk download model 70B)
ready = False
start_time = time.time()
last_status = 0

while time.time() - start_time < 1200:
    if vllm_proc.poll() is not None:
        vllm_log_f.flush()
        print("\n[ERROR] vLLM berhenti mendadak! 40 baris log terakhir:")
        with open(VLLM_LOG) as f:
            print("".join(f.readlines()[-40:]))
        raise RuntimeError("vLLM gagal start. Cek pesan error di atas (seringkali VRAM kurang atau MAX_MODEL_LEN terlalu besar).")

    try:
        r = requests.get(
            f"http://127.0.0.1:{VLLM_PORT}/v1/models",
            headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
            timeout=2,
        )
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass

    elapsed = int(time.time() - start_time)
    if elapsed - last_status >= 15:
        last_status = elapsed
        print(f"  ...masih memuat model ({elapsed} detik berlalu)")
    time.sleep(3)

if not ready:
    raise TimeoutError("Timeout menunggu vLLM siap. Cek /tmp/vllm.log di cell Step 6.")

print(f"\nvLLM READY dalam {int(time.time() - start_time)} detik! Membuka Cloudflare Tunnel...")

cf_log_f = open(CF_LOG, "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{VLLM_PORT}", "--no-autoupdate"],
    stdout=cf_log_f,
    stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    with open(CF_LOG) as f:
        content = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if match:
        PUBLIC_URL = match.group(0)
        break

if not PUBLIC_URL:
    with open(CF_LOG) as f:
        print(f.read())
    raise RuntimeError("Gagal mendapatkan URL Cloudflare Tunnel.")

API_BASE_URL = f"{PUBLIC_URL}/v1"

print("=" * 65)
print("API SERVER AKTIF & SIAP DIPAKAI!")
print("=" * 65)
print(f"Base URL   : {API_BASE_URL}")
print(f"API Key    : {VLLM_API_KEY}")
print(f"Model Name : {MODEL_ALIAS}")
print("=" * 65)
print("\nContoh curl:")
print(f'''curl {API_BASE_URL}/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer {VLLM_API_KEY}" \\
  -d '{{"model": "{MODEL_ALIAS}", "messages": [{{"role": "user", "content": "Halo!"}}]}}'
''')

In [ ]:
#@title 5. Tes Endpoint API (Lewat URL Publik)
import time, requests

TEST_PROMPT = "Halo! Perkenalkan dirimu secara singkat dalam bahasa Indonesia." #@param {type:"string"}

t0 = time.time()
resp = requests.post(
    f"{API_BASE_URL}/chat/completions",
    headers={
        "Authorization": f"Bearer {VLLM_API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "model": MODEL_ALIAS,
        "messages": [{"role": "user", "content": TEST_PROMPT}],
        "max_tokens": 256,
        "temperature": 0.7,
    },
    timeout=120,
)
resp.raise_for_status()
data = resp.json()
dt = time.time() - t0

usage = data.get("usage", {})
comp_tokens = usage.get("completion_tokens", 0)
tps = comp_tokens / dt if dt > 0 else 0

print(f"Status   : OK ({dt:.2f}s | {comp_tokens} tokens | ~{tps:.1f} tok/s)")
print("-" * 60)
print(data["choices"][0]["message"]["content"])

In [ ]:
#@title 6. (Opsional) Cek Log vLLM / Status Server
LINES = 40 #@param {type:"integer"}

print(f"--- {LINES} baris terakhir /tmp/vllm.log ---")
with open("/tmp/vllm.log") as f:
    print("".join(f.readlines()[-LINES:]))

In [ ]:
#@title 7. (Opsional) Matikan Server (Sebelum Ganti Model)
import subprocess

subprocess.run(["pkill", "-f", "vllm serve"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)
print("Server vLLM & Cloudflare Tunnel dihentikan.")
print("Ingat: Untuk menghentikan pemakaian Compute Units, klik Runtime -> Disconnect and delete runtime!")